In [ ]:
import pandas as pd
import os

def merge_health_datasets(file_list):
    """
    Merges multiple CSV files on common keys while removing redundant attributes.
    """
    # 1. Separate the static metadata from temporal (year-based) data
    temporal_files = []
    static_files = []

    print("Separating temporal and static files...")
    for f in file_list:
        df_temp = pd.read_csv(f, nrows=1)
        if 'year' in df_temp.columns:
            temporal_files.append(f)
        else:
            static_files.append(f)

    # 2. Initialize the merged dataframe with the first temporal file
    # We prioritize the first file as the 'base' for common attributes
    print("\nInitializing merged dataframe with the first temporal file...")
    merged_df = pd.read_csv(temporal_files[0])
    temporal_keys = ['country_code', 'year']
    print(f"  -> Initialized with {temporal_files[0]} (using keys: {temporal_keys})")

    # 3. Iteratively merge remaining temporal files
    print("\nMerging remaining temporal dataframes...")
    for file_name in temporal_files[1:]:
        df_next = pd.read_csv(file_name)

        # Identify redundant columns (already in merged_df and not a join key)
        redundant_cols = [c for c in df_next.columns
                          if c in merged_df.columns and c not in temporal_keys]

        # Drop redundant columns before merging
        df_next_cleaned = df_next.drop(columns=redundant_cols)

        # Perform outer join to keep all available data points
        merged_df = pd.merge(merged_df, df_next_cleaned, on=temporal_keys, how='outer')
        print(f"  -> Merged {file_name} (using keys: {temporal_keys}, dropped {len(redundant_cols)} redundant columns from current df)")

    # 4. Merge static metadata (e.g., longitude, latitude, capital)
    print("\nMerging static metadata dataframes...")
    for file_name in static_files:
        df_static = pd.read_csv(file_name)
        static_keys = ['country_code']

        redundant_cols = [c for c in df_static.columns
                          if c in merged_df.columns and c not in static_keys]

        df_static_cleaned = df_static.drop(columns=redundant_cols)
        merged_df = pd.merge(merged_df, df_static_cleaned, on=static_keys, how='left')
        print(f"  -> Merged {file_name} (using keys: {static_keys}, dropped {len(redundant_cols)} redundant columns from current df)")

    return merged_df

# List of files to merge
csv_files = [
    '/content/drive/MyDrive/VISDAT/Project/health_panel.csv', '/content/drive/MyDrive/VISDAT/Project/health_spending.csv', '/content/drive/MyDrive/VISDAT/Project/hospital_capacity.csv',
    '/content/drive/MyDrive/VISDAT/Project/infant_mortality.csv', '/content/drive/MyDrive/VISDAT/Project/life_expectancy.csv', '/content/drive/MyDrive/VISDAT/Project/ml_features.csv',
    '/content/drive/MyDrive/VISDAT/Project/context.csv', '/content/drive/MyDrive/VISDAT/Project/country_metadata.csv'
]

# Execute the merge
final_dataset = merge_health_datasets(csv_files)

# Save to a new CSV in the specified Drive folder
output_path_merged_data = '/content/drive/MyDrive/VISDAT/Project/merged_health_datafinal.csv'
final_dataset.to_csv(output_path_merged_data, index=False)

print(f"\nMerge Complete!")
print(f"Resulting Shape: {final_dataset.shape}")
print(f"Merged data saved to: {output_path_merged_data}")

Separating temporal and static files...

Initializing merged dataframe with the first temporal file...
  -> Initialized with /content/drive/MyDrive/VISDAT/Project/health_panel.csv (using keys: ['country_code', 'year'])

Merging remaining temporal dataframes...
  -> Merged /content/drive/MyDrive/VISDAT/Project/health_spending.csv (using keys: ['country_code', 'year'], dropped 8 redundant columns from current df)
  -> Merged /content/drive/MyDrive/VISDAT/Project/hospital_capacity.csv (using keys: ['country_code', 'year'], dropped 8 redundant columns from current df)
  -> Merged /content/drive/MyDrive/VISDAT/Project/infant_mortality.csv (using keys: ['country_code', 'year'], dropped 9 redundant columns from current df)
  -> Merged /content/drive/MyDrive/VISDAT/Project/life_expectancy.csv (using keys: ['country_code', 'year'], dropped 9 redundant columns from current df)
  -> Merged /content/drive/MyDrive/VISDAT/Project/ml_features.csv (using keys: ['country_code', 'year'], dropped 30 redu

In [ ]:
import pandas as pd

# Load the merged data from the Drive folder
merged_df = pd.read_csv('/content/drive/MyDrive/VISDAT/Project/merged_health_datafinal.csv')
display(merged_df.head())

,country_code,country_name,year,life_expectancy_total,life_expectancy_female,life_expectancy_male,life_expectancy_gender_gap,le_total_yoy_change,region,income_group,...,target_le_change_next_year,target_im_next_year,target_im_improvement,target_le_improves,target_le_5yr,target_le_gain_5yr,gdp_income_tier,capital,longitude,latitude
0,1A,Arab World,1960,44.794056,46.917488,42.850127,4.07,NaN,NaN,NaN,...,0.751,NaN,NaN,1.0,48.117159,3.323,NaN,NaN,NaN,NaN
1,1A,Arab World,1961,45.545371,47.709109,43.558405,4.15,0.751,NaN,NaN,...,0.448,NaN,NaN,1.0,48.625424,3.080,low_income,NaN,NaN,NaN
2,1A,Arab World,1962,45.993769,48.172152,44.005079,4.17,0.448,NaN,NaN,...,0.986,NaN,NaN,1.0,49.092434,3.099,low_income,NaN,NaN,NaN
3,1A,Arab World,1963,46.979338,48.881887,45.223308,3.66,0.986,NaN,NaN,...,0.591,NaN,NaN,1.0,49.657787,2.678,low_income,NaN,NaN,NaN
4,1A,Arab World,1964,47.570034,49.447872,45.830552,3.62,0.591,NaN,NaN,...,0.547,NaN,NaN,1.0,50.148857,2.579,low_income,NaN,NaN,NaN


In [ ]:
import pandas as pd
import numpy as np

def perform_data_cleaning(input_file, output_file):
    print(f"\nStarting data cleaning for {input_file}...")

    # 1. Load Data
    df = pd.read_csv(input_file)
    print(f"  -> Data loaded. Initial shape: {df.shape}")

    # 2. Remove entirely empty columns (usually non-matching metadata)
    initial_cols = df.shape[1]
    df = df.dropna(axis=1, how='all')
    cols_after_dropping_empty = df.shape[1]
    print(f"  -> Removed {initial_cols - cols_after_dropping_empty} entirely empty columns. New shape: {df.shape}")

    # 3. Sort for logical time-series cleaning
    if 'country_code' in df.columns and 'year' in df.columns:
        df = df.sort_values(by=['country_code', 'year'])
        print("  -> Data sorted by 'country_code' and 'year' for time-series cleaning.")
    else:
        print("  -> 'country_code' or 'year' not found, skipping sorting by these columns.")

    # 4. Numeric Imputation: Interpolate by Country
    num_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    if 'year' in num_cols: num_cols.remove('year')
    print(f"  -> Performing numeric imputation (interpolation) for {len(num_cols)} columns...")

    # Count missing values before interpolation
    missing_before_interpolation = df[num_cols].isnull().sum().sum()

    # Fill gaps within a country's timeline
    df[num_cols] = df.groupby('country_code')[num_cols].transform(
        lambda x: x.interpolate(method='linear', limit_direction='both')
    )
    missing_after_interpolation = df[num_cols].isnull().sum().sum()
    print(f"    -> Interpolated by country. {missing_before_interpolation - missing_after_interpolation} missing values filled.")

    # 5. Global Imputation (Fallback for countries with NO data for a metric)
    missing_before_global = df[num_cols].isnull().sum().sum()
    df[num_cols] = df[num_cols].fillna(df[num_cols].median())
    missing_after_global = df[num_cols].isnull().sum().sum()
    print(f"    -> Performed global median imputation for remaining numeric missing values. {missing_before_global - missing_after_global} missing values filled.")


    # 6. Categorical Imputation
    cat_cols = df.select_dtypes(include=['object']).columns.tolist()
    print(f"  -> Performing categorical imputation for {len(cat_cols)} columns...")
    missing_cat_before = df[cat_cols].isnull().sum().sum()
    for col in cat_cols:
        df[col] = df[col].fillna('Unknown').astype(str).str.strip()
    missing_cat_after = df[cat_cols].isnull().sum().sum()
    print(f"    -> Filled {missing_cat_before - missing_cat_after} missing categorical values with 'Unknown'.")

    # 7. Save Cleaned Data
    df.to_csv(output_file, index=False)
    total_missing_final = df.isnull().sum().sum()
    print(f"\nData Cleaning Complete. Missing values: {total_missing_final}")
    print(f"Cleaned data saved to: {output_file}")
    return df

# Run the program
cleaned_data = perform_data_cleaning('/content/drive/MyDrive/VISDAT/Project/merged_health_datafinal.csv', 'cleaned_health_data.csv')


Starting data cleaning for /content/drive/MyDrive/VISDAT/Project/merged_health_datafinal.csv...
  -> Data loaded. Initial shape: (17210, 173)
  -> Removed 6 entirely empty columns. New shape: (17210, 167)
  -> Data sorted by 'country_code' and 'year' for time-series cleaning.
  -> Performing numeric imputation (interpolation) for 160 columns...
    -> Interpolated by country. 890212 missing values filled.
    -> Performed global median imputation for remaining numeric missing values. 161720 missing values filled.
  -> Performing categorical imputation for 6 columns...
    -> Filled 29272 missing categorical values with 'Unknown'.

Data Cleaning Complete. Missing values: 0
Cleaned data saved to: cleaned_health_data.csv


In [ ]:
import pandas as pd

# Load the cleaned data to inspect it
cleaned_df = pd.read_csv('cleaned_health_data.csv')

print("First 5 rows of the cleaned data:")
display(cleaned_df.head())

print("\nInformation about the cleaned data (data types, non-null counts):")
cleaned_df.info()

First 5 rows of the cleaned data:


,country_code,country_name,year,life_expectancy_total,life_expectancy_female,life_expectancy_male,life_expectancy_gender_gap,le_total_yoy_change,health_spend_pct_gdp,health_spend_per_capita_usd,...,efficiency_score_roll5std,decade,target_le_next_year,target_le_change_next_year,target_im_next_year,target_im_improvement,target_le_improves,target_le_5yr,target_le_gain_5yr,gdp_income_tier
0,1A,Arab World,1960,44.794056,46.917488,42.850127,4.07,0.751,3.941685,111.859375,...,0.071,1960.0,45.545371,0.751,62.900376,-2.817,1.0,48.117159,3.323,Unknown
1,1A,Arab World,1961,45.545371,47.709109,43.558405,4.15,0.751,3.941685,111.859375,...,0.071,1960.0,45.993769,0.448,62.900376,-2.817,1.0,48.625424,3.080,low_income
2,1A,Arab World,1962,45.993769,48.172152,44.005079,4.17,0.448,3.941685,111.859375,...,0.071,1960.0,46.979338,0.986,62.900376,-2.817,1.0,49.092434,3.099,low_income
3,1A,Arab World,1963,46.979338,48.881887,45.223308,3.66,0.986,3.941685,111.859375,...,0.071,1960.0,47.570034,0.591,62.900376,-2.817,1.0,49.657787,2.678,low_income
4,1A,Arab World,1964,47.570034,49.447872,45.830552,3.62,0.591,3.941685,111.859375,...,0.071,1960.0,48.117159,0.547,62.900376,-2.817,1.0,50.148857,2.579,low_income



Information about the cleaned data (data types, non-null counts):
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 17210 entries, 0 to 17209
Columns: 167 entries, country_code to gdp_income_tier
dtypes: float64(160), int64(1), object(6)
memory usage: 21.9+ MB


In [ ]:
print("Column names in the cleaned data:")
display(cleaned_df.columns.tolist())

Column names in the cleaned data:


['country_code',
 'country_name',
 'year',
 'life_expectancy_total',
 'life_expectancy_female',
 'life_expectancy_male',
 'life_expectancy_gender_gap',
 'le_total_yoy_change',
 'health_spend_pct_gdp',
 'health_spend_per_capita_usd',
 'spend_pct_gdp_yoy_change',
 'spend_per_capita_yoy_pct',
 'hospital_beds_per_1000',
 'physicians_per_1000',
 'nurses_midwives_per_1000',
 'total_health_workers_per_1000',
 'infant_mortality_per_1000',
 'under5_mortality_per_1000',
 'maternal_mortality_per_100k',
 'infant_mortality_yoy_change',
 'under5_mortality_yoy_change',
 'gdp_per_capita_usd',
 'population_total',
 'log_gdp_per_capita',
 'gdp_per_capita_yoy_pct',
 'le_per_gdp_point',
 'le_per_1k_spend',
 'le_spend_residual',
 'efficiency_score',
 'spend_tier',
 'beds_capacity_tier',
 'infant_mortality_tier',
 'life_expectancy_total_lag1',
 'life_expectancy_total_lag2',
 'life_expectancy_total_lag3',
 'life_expectancy_total_lag4',
 'life_expectancy_total_lag5',
 'life_expectancy_female_lag1',
 'life_exp